<a href="https://colab.research.google.com/github/SridharS-Square/Agentic_AI_Workshop/blob/main/Building%20Advanced%20Al%20Agents%20with%20CrewAl/Automated_code_debugging_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install crewai langchain-google-genai

In [ ]:
# ai_code_review_workflow.py

import os
import logging
from crewai import Agent, Task, Crew
from langchain_google_genai import ChatGoogleGenerativeAI

# --- 1. Configuration and Setup ---

# Configure logging to display informational messages
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s] - %(message)s')

def setup_llm_provider():
    """Initializes and returns the Gemini LLM provider."""
    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        logging.error("The GOOGLE_API_KEY environment variable is not set.")
        raise ValueError("Please set the GOOGLE_API_KEY environment variable to run this script.")

    logging.info("Initializing Gemini 1.5 Flash LLM...")
    # Using a temperature > 0 helps prevent deterministic, repetitive outputs.
    gemini_llm = ChatGoogleGenerativeAI(
        google_api_key=api_key,
        model="gemini/gemini-1.5-flash",
        temperature=0.75
    )
    return gemini_llm

# --- 2. Input Data ---

# The Python code snippet that needs to be reviewed and fixed.
# Note: This version has intentional logic flaws for the agents to find.
code_to_review = """
def generate_fibonacci_sequence(count):
    # This function should generate a Fibonacci sequence of a specific length.
    if count <= 0:
        return "Count must be a positive integer."
    elif count == 1:
        return [0]

    sequence = [0, 1]
    # The loop condition is incorrect for the intended 'count'.
    while len(sequence) > count:
        next_val = sequence[-1] + sequence[-2]
        sequence.append(next_val)
    return sequence
"""

# --- 3. Define AI Agents ---

def define_review_agents(llm):
    """Defines and returns the code auditor and refactorer agents."""
    logging.info("Defining AI agents...")

    # Agent 1: The Auditor who finds issues
    code_auditor = Agent(
        role="Code Auditor",
        goal="Perform a detailed audit of a given Python code snippet to identify any logical errors, bugs, or deviations from best practices.",
        backstory="You are a meticulous and experienced software quality assurance engineer known as 'The Auditor.' Your sole focus is to scrutinize code with extreme precision, finding even the most subtle flaws before they can cause problems.",
        llm=llm,
        verbose=True,
        memory=False, # State is passed via tasks, no memory needed
    )

    # Agent 2: The Refactorer who fixes the code
    code_refactorer = Agent(
        role="Code Refactorer",
        goal="Rewrite and improve a Python code snippet to fix identified bugs and enhance its logic, clarity, and efficiency.",
        backstory="You are a pragmatic and skilled senior software developer known as 'The Refactorer.' You excel at transforming problematic code into clean, robust, and maintainable solutions based on expert analysis.",
        llm=llm,
        verbose=True,
        memory=False,
    )

    return code_auditor, code_refactorer

# --- 4. Define Tasks ---

def define_review_tasks(auditor_agent, refactorer_agent, code):
    """Defines and returns the analysis and correction tasks."""
    logging.info("Defining tasks for the agents...")

    # Task 1: Audit the code for errors
    audit_task = Task(
        description=f"""
        Audit the following Python code snippet. Your analysis must be thorough.
        Identify all logical errors, potential edge cases, and bugs.
        Present your findings as a clear, itemized list.

        Code to Audit:
        ```python
        {code}
        ```
        """,
        agent=auditor_agent,
        expected_output="A detailed, bullet-pointed list of all identified issues in the code."
    )

    # Task 2: Fix the code based on the audit
    refactor_task = Task(
        description=f"""
        Using the audit findings from the Code Auditor, rewrite the provided code snippet to correct all identified bugs.
        The final output should be ONLY the corrected, clean Python code block. Do not include any explanations or extra text.

        Original Flawed Code:
        ```python
        {code}
        ```
        """,
        agent=refactorer_agent,
        expected_output="The complete and corrected Python code as a single string, ready for execution.",
        context=[audit_task] # This task depends on the output of the audit_task
    )

    return audit_task, refactor_task

# --- 5. Main Execution ---

def main():
    """Main function to set up and run the CrewAI workflow."""
    try:
        llm_provider = setup_llm_provider()
        auditor, refactorer = define_review_agents(llm_provider)
        analysis_task, correction_task = define_review_tasks(auditor, refactorer, code_to_review)

        # Assemble the crew with a sequential workflow
        code_review_crew = Crew(
            agents=[auditor, refactorer],
            tasks=[analysis_task, correction_task],
            verbose=2, # Use verbose level 2 for detailed step-by-step logging
            process="sequential",
            memory=False
        )

        # Start the workflow
        logging.info("Starting the AI code review crew...")
        final_result = code_review_crew.kickoff()
        logging.info("Crew workflow has finished.")

        # Display the final, polished results
        print("\n" + "="*50)
        print("✅ AI Code Review Complete")
        print("="*50 + "\n")

        print("--- Agent Roles ---")
        print(f"🕵️  Auditor: {auditor.role}")
        print(f"🛠️  Refactorer: {refactorer.role}\n")

        print("--- Final Corrected Code ---")
        print(final_result)

        # You can also inspect the output of individual tasks if needed
        # print("\n--- Auditor's Raw Findings ---")
        # print(analysis_task.output.raw)

    except (ValueError, EnvironmentError) as e:
        logging.error(f"Configuration error: {e}")
    except Exception as e:
        logging.error(f"An unexpected error occurred during crew execution: {e}", exc_info=True)

if __name__ == "__main__":
    main()
